# Lesson 44 · Writing the STFT — give sound a time axis: framing + windowing + FFT by hand, matched point by point against aurora

**Goal**: first write out the complete three-step STFT pipeline: framing, windowing, FFT; then match the result point by point against `aurora.audio.stft.stft()` (`atol=1e-9`).

> **Implementation lesson**: pin down the shape `(n_frames, n_freq)` and the padding / frame-boundary policy first, then write the loop; for the conceptual trade-offs see L43.

🔗 **Aurora connection**
- What you rewrite: `src/aurora/audio/stft.py` — `stft()`, `frame_signal()`
- Test baseline: `tests/audio/test_transforms.py` — confirms the aurora FFT matches numpy to within 1e-9


← **Previous lesson** · [L43 · How the STFT works](L43_stft.ipynb)

> Last lesson covered **how the STFT works**: the short-time Fourier transform puts timestamps on a spectrum, with a time–frequency resolution trade-off.
> This lesson explores **writing the STFT** yourself.

## This lesson's story: how does music software show a spectrogram?

Open Audacity or Ableton, click "Spectrogram view", and you instantly see the frequency content of the music changing over time — the kick drum flashing at the bottom, the voice pulsing in the midrange, the cymbals flickering up at the top.

That picture is called a **spectrogram**, and the algorithm behind it is the STFT. The pipeline has just three steps:

```
1. frame_signal(x, win_len, hop)   → slice the signal into frames, shape (n_frames, win_len)
2. frames *= window                → multiply each frame by a Hann window to remove edge jumps
3. STFT = fft(frames, axis=1)      → FFT each frame, shape (n_frames, win_len//2+1)
```

Step 1 was built in L43, step 2 you mastered in L41; this lesson chains the three steps into one pipeline: `my_stft(x, win_len, hop)`.

**Intuition for the key parameters**:

| Parameter | Meaning | Effect |
|---|---|---|
| `win_len` | frame length (samples) | larger → finer frequency (Δf=sr/win_len); smaller → better time resolution |
| `hop` | frame step | smaller → frames closer together, clearer time detail |
| `window` | window function | Hann: sidelobes 31 dB lower; rect: leakage (blurs neighboring frequencies) |

Goal for this section: implement `my_stft` so that it matches `aurora.audio.stft.stft` to within 1e-9.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine, chirp


### 📌 The three core functions in this lesson

Before diving into the implementation, get to know the low-level tools Aurora provides:

| Function | What it does | Returns | Key parameters |
|---|---|---|---|
| `frame_signal(x, win_len, hop, center=True)` | slices a long signal into a frame matrix | shape `(n_frames, win_len)` | `center`: whether to reflect-pad |
| `get_window(name, n, periodic=True)` | builds a window function (Hann/Hamming/etc) | 1D ndarray, length `n` | `periodic`: True/False (changes the endpoint values) |
| `aurora_fft(x_frame)`, i.e. `from aurora.audio.transforms import fft` | hand-written Cooley-Tukey FFT (L37-39) | complex128 array | input length must be a power of 2 |

**Why must we use `aurora_fft` and not `np.fft.fft`?**

Aurora's core philosophy is "build it from scratch". That's not stubbornness — once the numerical precision is right and the logic is sound, whether or not you use numpy makes almost no difference to the answer (both are IEEE 754 floating point). But **the point of the teaching** is for you to derive and verify the FFT algorithm yourself, not to call a black box. When you later improve the FFT (say, by parallelizing it or using a specific precision), you can do so with confidence. `np.fft` hides every detail; you can't debug or improve it. So this lesson insists on `aurora_fft` — in later lessons you're free to mix the two.

**What is a "sidelobe" in a window function?**

The table mentions "Hann window: sidelobes 31 dB lower" — what does "sidelobe" mean here? Imagine analyzing a signal in the frequency domain with a rectangular window (rect, i.e. multiplying the signal by all ones). In theory, a pure 440 Hz sine wave should have energy in only one bin after the FFT, the 440 Hz bin. But with the rect window, the FFT result **leaks** energy into many other bins around 440 Hz, as if "ghost frequencies" appeared on both sides of 440 Hz. Those ghosts are the **sidelobes**. By tapering smoothly to 0 at the frame edges (instead of cutting off abruptly), the Hann window pushes the sidelobes down by 31 dB (roughly 1/35 of the amplitude, about 1/1000 of the power). That way, in frequency analysis, neighboring real frequency components don't contaminate each other.

This matters for sound analysis: if you want to find one particular instrument note in noisy music (say the A at 440 Hz), a window with low sidelobes lets you see the true energy distribution more clearly, instead of being fooled by leakage artifacts.

## Concept 1: the STFT pipeline

The core STFT formula:

```
STFT[t, f] = sum_{n=0}^{N-1}  x[t*hop + n] * w[n] * exp(-j*2*pi*f*n/N)
```

Here `N` is the window length (`win_len`), `w[n]` is the window function, `t` is the frame index, and `f` is the frequency-bin index. In code this is equivalent to: slice frames → multiply each frame by the window → take a length-N FFT of each frame → keep the first `N//2+1` columns.

`frame_signal(x, frame_length, hop_length)` slices the frames with a fancy-indexing trick, no Python loop:

```
idx[t, n] = t * hop + n      (t in [0, n_frames), n in [0, N))
frames = x[idx]               # shape: (n_frames, N)
```


#### 🔍 Formula breakdown: how the STFT goes from maths to code (a concrete N=8 example)

That formula looks scary, but it's really just three pieces multiplied together and summed. Let's take it apart with a tiny signal of only **8 samples**:

**Signal**: `x = [x₀, x₁, x₂, ..., x₇]` (8 samples)
**Window**: `w = [w₀, w₁, ..., w₇]` (a Hann window of length 8)
**Parameters**: `N=8, hop=4, t=0` (look at frame 0 only)

**The slice for frame 0**: `frame[0] = x[0:8] * w` (start at index 0, take 8 samples)

**The FFT's job**: compute how strong frame[0] is at **8 frequency components** (0 Hz, 1/N·sr, 2/N·sr, ..., 7/N·sr).

For frequency f=1 (meaning "frequency component number 1", which is sr/8 Hz), the STFT formula says:

```
STFT[0, f=1] = sum_{n=0}^{7}  (x[0+n] * w[n]) * exp(-j*2*pi*1*n/8)
             = (x[0]*w[0]) * exp(0)
             + (x[1]*w[1]) * exp(-j*2*pi*1/8)
             + (x[2]*w[2]) * exp(-j*2*pi*2/8)
             + ...
             + (x[7]*w[7]) * exp(-j*2*pi*7/8)
```

This is an **inner product** (a dot product): multiply the signal point by point with the **rotating complex exponential** `exp(-j*2*pi*f*n/N)` of one particular frequency, then add it all up. What is that `exp(...)` term? It's **a rotation on the unit circle** — a rotation of -2π·f·n/N radians in the complex plane. Different values of f rotate at different speeds, which is how each one can "pick out" the component at its own frequency from the signal.

**The essence**: the STFT sum is just the standard FFT definition; the only difference is:
- the FFT runs once over **the whole signal** (equivalent to t=0 with no framing)
- the STFT runs once on each of **many frames** (different t) and keeps every frame's spectrum

So the code is just a loop: `for t in range(n_frames): spectrum[t] = fft(x[t*hop : t*hop+N] * w)`.

**How does that mysterious complex exponential "pick out a frequency"?**

Picture `exp(-j*2*pi*f*n/N)` as a **sequence of rotating vectors** (rotating further as n increases). If some frequency component in the signal `frame[n]` oscillates exactly in step with this rotation (phase-aligned), then after the point-by-point multiplication all the vectors point in roughly the same direction, and their sum is as long as it can be. If the signal doesn't contain that frequency, the product vectors point every which way and add up to nearly 0. That's why the FFT can "fish out" one frequency from a mixed signal.

Picture it in the complex plane: the signal frame[n] can be decomposed into a sum of many sine waves at different frequencies. When you multiply by `exp(-j*2*pi*f*n/N)`, only the component whose frequency is exactly f gets amplified by "resonance"; the components at other frequencies cancel out when summed. That is the core idea of Fourier analysis.

Here's the N=8 toy example in code:

In [ ]:
import numpy as np
from aurora.audio.windows import get_window
from aurora.audio.transforms import fft as aurora_fft

# Toy example: a signal with N=8
N = 8
x_toy = np.array([1.0, 0.5, -0.3, 0.2, 0.8, -0.4, 0.1, 0.9], dtype=np.float32)  # 8 samples
w_toy = get_window("hann", N, periodic=True)

print("signal x:", x_toy)
print("Hann window w:", w_toy)
print()

# Apply the window
frame = x_toy * w_toy
print("after windowing (frame 0):", frame)
print()

# Compute the f=1 component with a hand-written "STFT formula sum" (as a check)
f_target = 1
stft_manual = 0 + 0j
for n in range(N):
    rot = np.exp(-1j * 2 * np.pi * f_target * n / N)  # twiddle factor
    stft_manual += frame[n] * rot
    print(f"  n={n}: frame[{n}]={frame[n]:.4f} × exp(-j*2π*{f_target}*{n}/{N}) → running sum {stft_manual:.4f}")

print()
print(f"hand-written sum STFT[0, f=1] = {stft_manual:.6f}")

# Get the result with the FFT
fft_result = aurora_fft(frame)
print(f"FFT result FFT[1]             = {fft_result[1]:.6f}")
print()

# Compare
print(f"Are they equal? {np.isclose(stft_manual, fft_result[1])}")
print("✅ The hand-written formula and the FFT agree! That confirms it: each STFT frame is just an FFT.")

In [ ]:
# Demo: framing
from aurora.audio.stft import frame_signal

sr = 16000
x = sine(440, 0.5, sample_rate=sr)          # 0.5 s, 440 Hz sine wave
win_len, hop = 2048, 512

frames = frame_signal(x, win_len, hop, center=True)
print(f"signal length: {len(x)} samples")
print(f"frame matrix shape: {frames.shape}")      # (n_frames, win_len)
print(f"n_frames = 1 + (padded_len - win_len) // hop")


## Concept 2: complex output and the magnitude spectrum

For a real signal the FFT outputs `N` complex numbers, but positive and negative frequencies are conjugate-symmetric, so the useful information is only in the first `N//2+1` bins (DC up to Nyquist).

```
spectrum = np.fft.fft(windowed_frame)        # shape: (N,)  complex
rfft_out = spectrum[:N//2+1]                 # shape: (N//2+1,)  keep positive frequencies only
```

Stacking all the frames, the STFT has shape `(n_frames, N//2+1)` and dtype `complex128`. The magnitude spectrum `magnitude = np.abs(stft_out)` throws away the phase and gives the strength of each time–frequency cell, with the same shape.


#### 🔍 Why N//2+1 and not N//2? (a picture of the FFT bins)

That "extra 1" confuses a lot of people. Let's look at the concrete case N=8:

The FFT outputs 8 complex numbers (indices 0-7), one for each of 8 frequency bins. For a **real signal**, the FFT has a property called **conjugate symmetry**:

```
FFT[k] = conj(FFT[N-k])
```

In other words, the value in bin k is the complex conjugate of the value in bin (N-k). That means the information is duplicated — if you know the first half, you can work out the second half.

**Specifically, for N=8**:
- bin 0 (DC / 0 Hz): `FFT[0]` — real (the sum)
- bin 1 (sr/8): `FFT[1]` — complex, carries information
- bin 2 (2·sr/8): `FFT[2]` — complex, carries information
- bin 3 (3·sr/8): `FFT[3]` — complex, carries information
- bin 4 (4·sr/8 = Nyquist): `FFT[4]` — **real** (no imaginary part)
- bin 5 (5·sr/8): `FFT[5]` — complex, but equal to `conj(FFT[3])`, redundant
- bin 6 (6·sr/8): `FFT[6]` — complex, but equal to `conj(FFT[2])`, redundant
- bin 7 (7·sr/8): `FFT[7]` — complex, but equal to `conj(FFT[1])`, redundant

So all we need are bins 0, 1, 2, 3, 4 — **5 bins** in total, which is `N//2 + 1 = 8//2 + 1 = 5`.

**Key point**:
- bin 0 (DC) is always real → counts as 1
- bin 1 to bin (N//2-1) → (N//2-1) complex numbers
- bin N//2 (Nyquist) is always real → counts as 1
- **Total**: 1 + (N//2-1) + 1 = **N//2 + 1** ✓

Here's the conjugate symmetry of an N=8 FFT in code:

In [ ]:
# Conjugate symmetry of an N=8 FFT
N = 8
fft_full = aurora_fft(x_toy * w_toy)  # all 8 bins

print(f"N={N}, full FFT output (8 bins):")
for k in range(N):
    print(f"  bin {k:d} (frequency {k:d}·sr/{N:d}): {fft_full[k]:.6f}")

print()
print("Checking conjugate symmetry: FFT[k] = conj(FFT[N-k])")
for k in range(1, N//2 + 1):
    conj_sym = np.conj(fft_full[N - k])
    is_close = np.isclose(fft_full[k], conj_sym)
    print(f"  FFT[{k}] = {fft_full[k]:.6f}")
    print(f"  conj(FFT[{N-k}]) = {conj_sym:.6f}")
    print(f"  match? {is_close} ✓" if is_close else f"  match? {is_close} ✗")
    print()

print(f"Keeping just the first N//2+1 = {N//2+1} bins is enough (lossless):")
rfft_out = fft_full[:N//2 + 1]
print(f"  rfft = {rfft_out}")
print(f"  these {len(rfft_out)} values hold all the information; the remaining {N - len(rfft_out)} bins are redundant")

In [ ]:
# Demo: one frame's FFT and its complex output
from aurora.audio.windows import get_window

win = get_window("hann", win_len, periodic=True)   # periodic Hann window, same as numpy's convention
frame0 = frames[0] * win                           # window frame 0

full_fft = np.fft.fft(frame0)                      # full FFT, shape (win_len,)
rfft_out = full_fft[:win_len // 2 + 1]            # keep the positive-frequency part

print(f"full_fft shape: {full_fft.shape}")
print(f"rfft_out shape: {rfft_out.shape}")         # (1025,) for win_len=2048
print(f"rfft_out dtype: {rfft_out.dtype}")         # complex128
print(f"magnitude[0:5]: {np.abs(rfft_out[:5])}")


#### 🔍 `periodic=True` vs `False`: how the Hann window differs at its endpoints

Above we used "the periodic Hann window, same as numpy's convention" — so what exactly does the `periodic` parameter change?

**The intuitive picture**: the Hann window has to taper smoothly to 0 at the frame edges. With `periodic=False` (the symmetric window), the window is exactly 0 at both ends. With `periodic=True`, the window is one period of a slightly longer symmetric window with its last point dropped: `w[0]` is still exactly 0, but `w[-1]` is a little above 0 (for `win_len=8` it's about 0.146). That seemingly tiny difference matters for overlap-add of neighboring frames.

**Why does Aurora choose `periodic=True`?** Because when you frame with `frame_signal` and rebuild the signal by `overlap-add`, a `periodic=True` window guarantees that neighboring frames add up to a constant (they compensate for each other), with no "dips between frames". This is the standard practice in audio processing.

Here's a side-by-side comparison in code:

In [ ]:
# periodic=True vs False comparison
from aurora.audio.windows import get_window

win_len = 8
w_periodic = get_window("hann", win_len, periodic=True)
w_symmetric = get_window("hann", win_len, periodic=False)

print(f"win_len = {win_len}")
print()
print("periodic=True (Aurora's default):")
print(f"  {w_periodic}")
print(f"  endpoints: w[0]={w_periodic[0]:.6f}, w[-1]={w_periodic[-1]:.6f}")
print()
print("periodic=False (symmetric window):")
print(f"  {w_symmetric}")
print(f"  endpoints: w[0]={w_symmetric[0]:.6f}, w[-1]={w_symmetric[-1]:.6f}")
print()

# Plot the comparison
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(w_periodic, 'o-', label='periodic=True', linewidth=2, markersize=6)
ax.plot(w_symmetric, 's-', label='periodic=False', linewidth=2, markersize=6, alpha=0.7)
ax.set_xlabel('sample index n')
ax.set_ylabel('window value')
ax.set_title('Hann window: periodic=True vs False (the difference at the endpoints)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice: with periodic=True the last point is above 0, so neighboring frames add up to a constant; with periodic=False both endpoints are exactly 0.")
print("For overlap-add reconstruction, periodic=True is the standard choice.")

## Concept 3: center=True mode — centering frame 0 on x[0]

With `center=False`, frame 0 starts at `x[0]` and its window center sits at `x[win_len//2]`, so the energy at the very start of the signal gets squashed almost to nothing.

`center=True` pads `win_len//2` samples onto each end of the signal (Aurora uses reflect padding), so that the window center of frame `t` lands exactly on `x[t * hop]`:

```
pad = win_len // 2
x_padded = np.pad(x, pad, mode="reflect")
# frame 0: x_padded[0 : win_len]  → centered at x_padded[pad] = x[0]
# frame t: x_padded[t*hop : t*hop + win_len]
```

Reflect padding lets the first and last frames be analyzed in full, and it's the default behavior in librosa and Aurora. It mirrors the signal across the boundary, so unlike zero padding it doesn't conjure up an extra amplitude ramp at the boundary (a boundary transient).


#### 🔍 Zero padding vs reflect padding: where does the "amplitude ramp" come from?

That "amplitude ramp" is hard to picture. Let's look at the start of an actual signal:

Suppose we have a sine wave, but **the signal starts abruptly from some middle value** (say it starts at 0.8 rather than 0) and then heads downward.

What **zero padding** does: pad the start with zeros
```
original (start):  [0.8, 0.6, 0.3, 0.1, -0.2, -0.5, ...]
zero-padded (pad=2): [0, 0, 0.8, 0.6, 0.3, 0.1, -0.2, -0.5, ...]
                      ^ gap ^
```

This creates a **step** where the padding meets the signal: a sudden jump from 0 to 0.8. In the frequency domain a step shows up as high-frequency content (a bit of energy at every frequency), as if a steep "ramp" were being used to approximate the step.

What **reflect padding** does: mirror the signal
```
original (start):    [0.8, 0.6, 0.3, 0.1, -0.2, -0.5, ...]
reflect-padded (pad=2): [0.3, 0.6, 0.8, 0.6, 0.3, 0.1, -0.2, -0.5, ...]
                      ^mirror ^  ^original starts
```

The padded part `[0.3, 0.6]` is the two samples right after the start, `[0.6, 0.3]`, mirrored around the first sample 0.8. **The key**: the padding joins the original signal smoothly (no step), so no high-frequency "noise" (artifact) appears at the seam.

Let's see the effect in code and plots:

#### 🔍 What "opposite phase" means, as a waveform

The center=True demo below says that "the reflected left half of frame 0 is opposite in phase to the right half". If you haven't studied complex numbers that sounds abstract, so let me explain it in terms of waveforms.

Picture a signal that begins as a rising sine wave: `[0.1, 0.3, 0.5, 0.8, ...]`. After reflect padding it becomes: `[..., 0.5, 0.3, 0.1, | 0.1, 0.3, 0.5, 0.8, ...]` (`|` marks where the original signal starts).

The padded part `[0.5, 0.3, 0.1]` is a mirror image — it's **in reverse order**, like playing the waveform "backwards". When the two parts are multiplied by a Hann window together and then FFT'd:

- the original part: `0.1` gets the smallest window value, `0.8` a medium one
- the mirrored part: `0.5` gets the smallest window value, `0.1` a medium one

This mirror image runs **opposite in phase (the timing of the waveform)** to the original signal. In the complex plane, if one frequency component of the original signal points "northeast" (some angle in the complex plane), the same-frequency component of the mirrored part points "southwest" (the opposite angle). Two vectors in opposite directions cancel when added (like two equal and opposite forces).

That's why "opposite phase → energy cancels". This is not the "uncertainty principle" of quantum mechanics — there's no measurement uncertainty here at all. It's pure wave superposition: two sine waves with the same frequency and amplitude but opposite phase add up to nearly 0.

Here's that superposition in code:

In [ ]:
# Superposition demo: how opposite phase leads to energy cancellation
import numpy as np
import matplotlib.pyplot as plt

# Build two sine waves with opposite phase
t = np.linspace(0, 1, 1000, endpoint=False)
freq = 5  # 5 Hz
amp = 1.0

# Original wave
wave1 = amp * np.sin(2 * np.pi * freq * t)

# The opposite-phase (inverted) wave
wave2 = -amp * np.sin(2 * np.pi * freq * t)

# Superpose
wave_sum = wave1 + wave2

# Plot
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

ax = axes[0]
ax.plot(t, wave1, 'b-', linewidth=2, label='wave 1: sine wave')
ax.set_ylabel('amplitude')
ax.set_title('A single waveform')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 0.5])

ax = axes[1]
ax.plot(t, wave2, 'r-', linewidth=2, label='wave 2: inverted sine (180° out of phase)')
ax.set_ylabel('amplitude')
ax.set_title('The opposite-phase waveform')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 0.5])

ax = axes[2]
ax.plot(t, wave1, 'b-', linewidth=2, label='wave 1', alpha=0.7)
ax.plot(t, wave2, 'r-', linewidth=2, label='wave 2', alpha=0.7)
ax.plot(t, wave_sum, 'g-', linewidth=3, label='wave 1 + wave 2 = 0 (complete cancellation)')
ax.set_xlabel('time (s)')
ax.set_ylabel('amplitude')
ax.set_title('Two opposite-phase waves superposed → cancellation')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 0.5])

plt.tight_layout()
plt.show()

print("Superposition demo:")
print(f"  wave 1 (blue) + wave 2 (red) = 0 (green, a flat line at zero)")
print(f"  max amplitude: {np.abs(wave_sum).max():.2e} (zero to machine precision)")
print()
print("This is what happens in frame 0 with center=True:")
print("  • the left half (reflected mirror) is opposite in phase to the right half (original signal)")
print("  • same-frequency components cancel, and the total energy drops sharply")
print("  • so the peak is lower than with center=False")

In [ ]:
import matplotlib.pyplot as plt

# Build a signal that "doesn't start from 0"
x_test = np.sin(2 * np.pi * 440 * np.arange(200) / 16000).astype(np.float32)
# Show only the first 20 samples
x_head = x_test[:20]

# Padding amount
pad = 4

# Zero padding
x_zero_pad = np.pad(x_head, pad, mode='constant', constant_values=0)

# Reflect padding
x_reflect_pad = np.pad(x_head, pad, mode='reflect')

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Zero padding
ax = axes[0]
x_idx = np.arange(len(x_zero_pad))
ax.plot(x_idx, x_zero_pad, 'o-', linewidth=2, markersize=4, label='padded signal')
ax.axvline(pad - 0.5, color='red', linestyle='--', linewidth=1.5, label='padding / signal seam')
ax.axvline(pad + len(x_head) - 0.5, color='green', linestyle='--', linewidth=1.5, label='end of original signal')
ax.set_title('Zero padding: a step at the boundary → high-frequency artifact', fontsize=12, fontweight='bold')
ax.set_ylabel('amplitude')
ax.legend()
ax.grid(True, alpha=0.3)

# Reflect padding
ax = axes[1]
ax.plot(x_idx, x_reflect_pad, 'o-', linewidth=2, markersize=4, color='orange', label='padded signal')
ax.axvline(pad - 0.5, color='red', linestyle='--', linewidth=1.5, label='padding / signal seam')
ax.axvline(pad + len(x_head) - 0.5, color='green', linestyle='--', linewidth=1.5, label='end of original signal')
ax.set_title('Reflect padding: smooth join, no step → no high-frequency artifact', fontsize=12, fontweight='bold')
ax.set_ylabel('amplitude')
ax.set_xlabel('sample index')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Compare the two plots:")
print("  zero padding: a visible kink where the padding (left) meets the signal (right), at the red line it jumps from ~0 to the signal value")
print("  reflect padding: the padding flows smoothly into the original signal (no kink)")
print()

# Going further: how the two kinds of padding look in the FFT frequency domain
win = get_window("hann", len(x_head), periodic=True)
spec_zero = np.abs(np.fft.fft(x_zero_pad * np.pad(win, pad, mode='constant')))[:len(x_zero_pad)//2]
spec_reflect = np.abs(np.fft.fft(x_reflect_pad * np.pad(win, pad, mode='reflect')))[:len(x_reflect_pad)//2]

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(spec_zero, label='zero padding', linewidth=2)
ax.semilogy(spec_reflect, label='reflect padding', linewidth=2)
ax.set_title('Frequency-domain comparison: reflect padding gives a cleaner spectrum (less high-frequency energy)', fontsize=12, fontweight='bold')
ax.set_ylabel('magnitude (log scale)')
ax.set_xlabel('frequency bin')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("Frequency-domain comparison:")
print(f"  high-frequency energy with zero padding:    {spec_zero[100:].mean():.6f}")
print(f"  high-frequency energy with reflect padding: {spec_reflect[100:].mean():.6f}")
print("  reflect padding wins (lower energy), meaning fewer boundary artifacts")

In [ ]:
# Demo: the effect of center=True padding
# Restore this lesson's parameters (the comparison cells above set win_len to a small value)
win_len, hop = 1024, 256
win = get_window("hann", win_len, periodic=True)
pad = win_len // 2
x_padded = np.pad(x, pad, mode="reflect")

print(f"original signal length: {len(x)}")
print(f"length after reflect padding: {len(x_padded)}")
print(f"center of frame 0 (index after padding): {pad}  → corresponds to x[0]")

# Unpadded vs padded: peak magnitude of frame 0
frames_no_center = frame_signal(x, win_len, hop, center=False)
frames_center    = frame_signal(x, win_len, hop, center=True)

mag_no = np.abs(np.fft.fft(frames_no_center[0] * win)[:win_len//2+1])
mag_ctr = np.abs(np.fft.fft(frames_center[0]    * win)[:win_len//2+1])

print(f"center=False frame 0 peak: {mag_no.max():.4f}")
print(f"center=True  frame 0 peak: {mag_ctr.max():.4f}")
print()

print("📌 Understanding the center=True trade-off:")
print()
print("✓ What center=True buys you:")
print("  • the window center of frame t sits exactly on x[t*hop], so time alignment is exact")
print("  • the start (x[0]) and end (x[-1]) of the signal are analyzed in full, not faded out by the window")
print("  • the resulting STFT time axis is more trustworthy — every point in time has a frame centered on it")
print()

print("✗ What center=True costs you:")
print("  • the left half of frame 0 is a reflected mirror, opposite in phase to the right half")
print("  • opposite-phase signals partly cancel when added, so the peak actually comes out lower")
print("  • but this isn't 'lost energy', just cancellation — the total energy is conserved, only its distribution changes")
print()

print("🎯 Which to choose:")
print("  • music analysis, speech recognition → use center=True (time alignment matters more than peak height)")
print("  • energy detection, simple thresholds → center=False (keeps the largest peak)")
print("  • Aurora defaults to center=True, which is also librosa's standard choice")
print()

print("Note: with center=True the left half of frame 0 is a reflected mirror; the two halves are opposite in phase and partly cancel, so the peak is lower.")
print("      The point of center=True isn't the energy level but time alignment: the window center of frame t sits exactly on x[t*hop].")

## 1. ✏️ Implement `my_stft(x, win_len=2048, hop=512, window="hann")`

**Three steps chained together**:

| Step | Code | Notes |
|---|---|---|
| 1 | `frames = frame_signal(x, win_len, hop)` | slice frames, shape (n_frames, win_len) |
| 2 | `frames *= get_window(window, win_len)` | apply the window, the same window vector for every frame |
| 3 | per frame `aurora_fft(f)[:win_len//2+1]`, then `np.stack` | the iron rule: the engine is your hand-written FFT from L37-39 (win_len=2048 is a power of 2); `np.fft` is only a reference to compare against |

**Acceptance criteria**:
- output shape = `(n_frames, win_len//2+1)` (one-sided spectrum)
- `np.allclose(my_stft(x, win_len, hop), aurora_stft(x, win_len, hop), atol=1e-9)`
- `win_len=2048, hop=512, sr=16000, duration=0.5` (8000 samples) → aurora defaults to `center=True`, so n_frames = 1 + 8000//512 = 16 (only with `center=False` would it be 1+(8000-2048)//512 = 12)

**Imports you can use**:
```python
from aurora.audio.stft import frame_signal
from aurora.audio.windows import get_window
from aurora.audio.transforms import fft as aurora_fft  # Cooley-Tukey FFT built from scratch
```

## Common failure modes (read these before writing my_stft)

- Forgetting `rfft` / to keep only `win_len//2+1` bins → shape twice as big, or alignment fails
- With `center=True` your frame count is off by a few from the formula you worked out → **trust `aurora.stft.stft`**
- FFT-ing the frames without the window → sidelobe leakage explodes (L36)
- Swapping `hop` and `win_len` → n_frames is completely wrong

Stuck → go back to L43 `frame_signal` / L41 windowing / L39 FFT; the full reference is in `solutions/L44_stft_implement_solutions.md`.


## Half-step exercise · a single-frame STFT (not a substitute for my_stft)

First get it working for **frame 0** only: `frame_signal` → `× hann` → `rfft` → magnitude.
Once it matches row 0 of `aurora.stft`, write the `for` loop to assemble the full `my_stft`.


### Step-by-step guide: the function calls inside my_stft

You'll use these functions in both the half-step exercise and the full implementation. Make sure you understand each function's **input and output**:

```python
# 1. Build the window
win = get_window("hann", win_len, periodic=True)
#     input: window name "hann", length win_len (int), periodic mode
#     output: 1D numpy array, shape (win_len,), dtype float64

# 2. Slice frames
frames = frame_signal(x, win_len, hop, center=True)
#        input: signal x (1D array), frame length win_len (int), hop (int)
#        output: 2D numpy array, shape (n_frames, win_len)

# 3. Window every frame
windowed = frames * win
#          input: frame matrix (n_frames, win_len) × window vector (win_len,)
#          output: 2D array, shape (n_frames, win_len) (broadcast multiply)

# 4. FFT each frame
spec = aurora_fft(windowed[0])  # example for frame 0
#      input: 1D array (one frame), length must be a power of 2 (2048 etc.)
#      output: 1D complex128 array, shape (win_len,), holding win_len complex numbers

# 5. Keep the first win_len//2+1 columns (the rfft part)
rfft_out = spec[:win_len//2 + 1]
#          input: full FFT result (win_len complex numbers)
#          output: 1D complex128 array, shape (win_len//2+1,)

# 6. Stack every frame's STFT
stft_matrix = np.stack([aurora_fft(windowed[t])[:win_len//2+1] for t in range(n_frames)], axis=0)
#             output: 2D complex128 array, shape (n_frames, win_len//2+1)
```

**Common slip-ups**:
- ❌ Forgetting `[:win_len//2+1]` → output shape (n_frames, win_len) instead of (n_frames, win_len//2+1)
- ❌ Window vector with the wrong shape → shapes don't line up when broadcasting
- ❌ Using `np.fft.fft()` instead of `aurora_fft()` → breaks the "iron rule"

Here's the single-frame exercise:

In [ ]:
import numpy as np
from aurora.audio.io import sine
from aurora.audio.stft import frame_signal, stft
from aurora.audio.windows import get_window
from aurora.audio.transforms import fft as aurora_fft  # hand-written FFT engine (L37-39)

sr = 16_000
x = sine(440.0, 0.5, sample_rate=sr)
win_len, hop = 2048, 512

# Step 1: slice frames
frames = frame_signal(x, win_len, hop, center=True)
print(f"step 1 - frames shape: {frames.shape}  # (n_frames, win_len)")

# Step 2: build the Hann window
win = get_window("hann", win_len, periodic=True)
print(f"step 2 - win shape: {win.shape}  # (win_len,)")

# Step 3: window frame 0
frame0_windowed = frames[0] * win
print(f"step 3 - frame0_windowed shape: {frame0_windowed.shape}  # (win_len,)")

# Step 4: FFT frame 0 (full FFT)
spec0_full = aurora_fft(frame0_windowed)
print(f"step 4 - spec0_full shape: {spec0_full.shape}  # (win_len,) complex")

# Step 5: keep the first win_len//2+1 columns (the rfft part)
spec0 = spec0_full[:win_len//2+1]
print(f"step 5 - spec0 shape: {spec0.shape}  # (win_len//2+1,) complex")

# Check that it matches row 0 of aurora.stft
ref = stft(x, n_fft=win_len, hop_length=hop, window="hann")
print(f"\nCompared with aurora.stft:")
print(f"  ref[0] shape: {ref[0].shape}")
print(f"  spec0 close to ref[0]? {np.allclose(spec0, ref[0], atol=1e-9)}")

if np.allclose(spec0, ref[0], atol=1e-9):
    print("\n✅ Single frame matches! Now you see it:")
    print("  frame_signal → windowed frames → aurora_fft → [:win_len//2+1] = one row of the STFT")
    print("  Next: loop over every frame and stack them into a 2D matrix — that's the full my_stft")
else:
    print(f"\n❌ Values don't match, max error = {np.abs(spec0 - ref[0]).max():.2e}")

In [ ]:
def my_stft(x, win_len=2048, hop=512, window="hann"):
    """Hand-written STFT; returns a complex128 array of shape (n_frames, win_len//2+1)."""
    from aurora.audio.windows import get_window
    from aurora.audio.stft import frame_signal
    from aurora.audio.transforms import fft as aurora_fft  # hand-written Cooley-Tukey FFT from L37-39

    # ✏️ TODO 1: build the window win, shape (win_len,), periodic=True
    raise NotImplementedError("TODO: build the window with get_window(window, win_len, periodic=True)")

    # ✏️ TODO 2: slice frames, shape (n_frames, win_len), center=True
    raise NotImplementedError("TODO: frame the signal, shape=(n_frames, win_len)")

    # ✏️ TODO 3: window every frame
    raise NotImplementedError("TODO: windowed = frames * win")

    # ✏️ TODO 4: FFT each frame, keep the first win_len//2+1 columns
    n_bins = win_len // 2 + 1
    spectra = []
    for f in windowed:
        pass  # ✏️ TODO 4: append aurora_fft(f)[:n_bins] to spectra (iron rule: the engine is your hand-written FFT; np.fft is only for comparison)

    # ✏️ TODO 5: stack into a 2D array and return it
    return None


In [ ]:
from aurora.audio.stft import stft

sr = 16000
x = sine(440, 0.5, sample_rate=sr)

try:
    out = my_stft(x, win_len=2048, hop=512, window="hann")
except (NotImplementedError, TypeError):
    print("⬜ Finish the my_stft exercise above first, then run this cell")
else:
    ref = stft(x, n_fft=2048, hop_length=512, window="hann")

    assert out is not None, "not implemented yet: it returned None"
    assert out.shape == ref.shape, f"shape mismatch: got {out.shape}, expected {ref.shape}"
    assert out.dtype == np.complex128, f"dtype should be complex128, got {out.dtype}"
    assert np.allclose(out, ref, atol=1e-9), (
        f"values don't match, max error {np.abs(out - ref).max():.2e}"
    )
    print(f"✅ shape {out.shape}, dtype {out.dtype}, max_err < 1e-9")


## Parameter experiment: how `win_len` shapes the spectrogram

Fix `hop=256` and compare three settings, `win_len = 128 / 512 / 2048`:

| `win_len` | Frequency resolution `sr/win_len` | Time step `hop/sr` | What to expect |
|---|---|---|---|
| 128 | 125 Hz/bin | 16 ms | fine in time, blurry in frequency — the 440 Hz sine peak is very wide |
| 512 | 31 Hz/bin | 16 ms | a middle-of-the-road balance |
| 2048 | 7.8 Hz/bin | 16 ms | sharp in frequency — the 440 Hz peak covers only 1-2 bins |

The bigger `win_len`, the finer the frequency resolution, but the longer each frame lasts (so time resolution drops). This is a direct consequence of the STFT's **time–frequency uncertainty** (the Heisenberg-Gabor limit).


### Going deeper: frequency resolution vs time resolution

How do we get the **frequency resolution** Δf = sr / win_len?

When we FFT a frame of length win_len, we get win_len complex numbers. How many frequency points does that correspond to? The answer: from 0 Hz up to sr/2 Hz (Nyquist), the bins are spaced evenly, win_len/2 intervals in all (for a real signal).

```
frequency range: 0 Hz ~ sr/2 Hz
number of intervals: win_len/2
frequency resolution = (sr/2) / (win_len/2) = sr / win_len Hz/bin
```

Which means:
- `win_len = 2048, sr = 16000` → Δf = 16000/2048 ≈ 7.8 Hz/bin
- if you want to tell 440 Hz from 441 Hz, you need win_len ≥ sr/1 = 16000 (way too big)
- but if you only need to tell 440 Hz from 450 Hz (10 Hz apart), win_len ≥ sr/10 = 1600 is enough

What is **time resolution**? There are two ideas here; don't mix them up:

1. **Frame spacing** (hop/sr): how far apart two neighboring frames are. This sets how densely the time axis is sampled.
   - hop = 512, sr = 16000 → frame spacing = 512/16000 = 32 ms
   - if you want to see "how a phoneme changes within 10 ms", that spacing is too coarse; you need a smaller hop

2. **Frame duration** (win_len/sr): how much signal each frame covers. This is what ties into frequency resolution.
   - win_len = 2048, sr = 16000 → each frame covers 2048/16000 = 128 ms
   - within those 128 ms you can't see fast frequency changes (a jump from 400 Hz to 500 Hz within 40 ms, say, just gets averaged away)

**Time–frequency uncertainty (the Heisenberg-Gabor limit)**:
- if win_len ↑, frequency resolution ↑ (Δf ↓), but frame duration ↑ and time resolution ↓ — you can't see fast frequency changes
- if win_len ↓, time resolution ↑ (you can see fast changes), but frequency resolution ↓ — you can't tell apart the energy in neighboring frequency bins

This is a fundamental limit of Fourier analysis, not a flaw in the algorithm: **frame duration × Δf is essentially constant, so you can't optimize both at once**. The table below shows the trade-off:

| `win_len` | Frequency resolution `sr/win_len` | Frame duration `win_len/sr` | What you can see |
|---|---|---|---|
| 128 | 125 Hz/bin | 8 ms | fast changes (but mushy frequency) — good for watching quick bursts of loudness |
| 512 | 31 Hz/bin | 32 ms | a middle-of-the-road balance |
| 2048 | 7.8 Hz/bin | 128 ms | sharp frequency (but mushy time) — good for watching how steady a musical pitch is |

Let's check the trade-off in code:

In [ ]:
# Computing and checking the time–frequency resolution trade-off
sr = 16000
hop = 256

print("Time–frequency uncertainty (Heisenberg-Gabor) in concrete numbers:")
print("=" * 60)
for win_len in [128, 512, 2048]:
    delta_f = sr / win_len  # frequency resolution
    frame_time = win_len / sr * 1000  # frame duration (ms)
    hop_time = hop / sr * 1000  # frame spacing (ms)
    time_bandwidth_product = delta_f * frame_time  # time–bandwidth product

    print(f"\nwin_len = {win_len}:")
    print(f"  frequency resolution Δf = {sr}/{win_len} = {delta_f:.2f} Hz/bin")
    print(f"  frame duration = {win_len}/{sr}×1000 = {frame_time:.1f} ms (the width of the 'window' the FFT sees)")
    print(f"  frame spacing = {hop}/{sr}×1000 = {hop_time:.2f} ms (the sampling interval of the time axis)")
    print(f"  time–bandwidth product ≈ {time_bandwidth_product:.0f} (constant: it measures the size of the uncertainty)")
    print(f"  → smallest frequency difference you can resolve ≥ {delta_f:.2f} Hz")
    print(f"  → you can't see time detail finer than {frame_time:.0f} ms")

print("\n" + "=" * 60)
print("Conclusion:")
print("  • win_len ↑ → clear in frequency, mushy in time (good for musical notes / steady signals)")
print("  • win_len ↓ → clear in time, mushy in frequency (good for percussion / fast changes)")

In [ ]:
sr = 16000
x_chirp = chirp(200, 4000, 1.0, sample_rate=sr)   # 1 s linear chirp: 200 → 4000 Hz
hop = 256

# ⚠️ Guard: skip the plots while my_stft isn't implemented yet (raises NotImplementedError or returns None)
try:
    _guard = my_stft(x_chirp, win_len=512, hop=hop)
except (NotImplementedError, TypeError):
    _guard = None
if _guard is None:
    print("⚠️  my_stft isn't implemented yet, skipping the plots — rerun this cell after the exercise")
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for ax, wl in zip(axes, [128, 512, 2048]):
        S = np.abs(my_stft(x_chirp, win_len=wl, hop=hop)) ** 2
        S_db = 10 * np.log10(S.T + 1e-8)              # to dB; transpose so frequency is on the y axis

        im = ax.imshow(
            S_db,
            origin="lower",
            aspect="auto",
            extent=[0, len(x_chirp)/sr, 0, sr/2],
            vmin=-60, vmax=0,
            cmap="inferno",
        )
        ax.set_title(f"win_len={wl}  ({sr/wl:.0f} Hz/bin)")
        ax.set_xlabel("time (s)")
        ax.set_ylabel("frequency (Hz)")

    plt.colorbar(im, ax=axes, label="dB")
    plt.suptitle("STFT parameter experiment: how win_len affects time–frequency resolution", fontsize=13)
    plt.tight_layout()
    plt.show()


### Standard preprocessing for spectrum plots: from complex numbers to dB

Before drawing a spectrogram, the plotting code transforms the STFT like this:
```python
S = np.abs(my_stft(...)) ** 2          # 1. squared magnitude → power spectrum
S_db = 10 * np.log10(S + 1e-8)         # 2. log10 → decibels (dB)
```

Why these three steps?

**Step 1: squaring (`** 2`)**

`aurora_fft()` returns complex numbers and `np.abs()` takes the modulus (the magnitude) — so why square it too?

In signal processing, **the square of the amplitude is proportional to energy**. If you have a 1 V sine wave, its amplitude is 1 and its power (energy density) is 1²=1. The vertical axis of a spectrum usually shows power rather than amplitude, which fits our intuition of "how strong is this frequency" better. That's why `|X[k]|²` is called the "power spectrum".

**Step 2: log10 and decibels (dB)**

Power spectrum values span an enormous range (from 1e-10 to 1000+). On a linear scale the small values vanish and the big ones go off the chart. Taking log10 squeezes those 10+ orders of magnitude into a range that's easier to look at (typically -60 dB to 0 dB).

The formula:
```
dB = 10 × log10(power)
```

We multiply by 10 (not 20) here because we've already squared. If you took dB directly from the amplitude, you'd use `20 × log10(amplitude)`.

**Step 3: adding a small constant (`+ 1e-8`)**

Why add 1e-8? To keep `log10(0)` from being undefined. In practice:
- if the power somewhere is 0 (in theory it won't be, but floating-point noise can bring it very close to 0), `log10(0)` gives -∞
- after adding 1e-8, the minimum is `10 × log10(1e-8) = 10 × (-8) = -80 dB`
- so the darkest (minimum) value in the plot sits around -80 dB, which is sensible

**The intuitive picture**:
- dB is a logarithmic way of writing "a ratio relative to some reference"
- in audio, dB ≈ "relative loudness as the ear perceives it" (+20 dB is 10× the amplitude, 100× the power)
- power spectrum in dB = 10 × log10(power): a tone whose power goes up 10× goes up 10 dB

You saw this conversion at work in the parameter experiment above.

## 🎯 The payoff later
The `my_stft` you assembled yourself today (framing → windowing → FFT) will serve, unchanged, as the first stage of the pipeline in **L45 spectrograms / L47 log-mel / L50 MFCC** — every audio feature after this grows out of this time–frequency matrix. The pain of debugging `center` and windowing in this lesson saves you time in the next five.

## Wrap-up

`my_stft` fully reproduces Aurora's STFT computation: first `frame_signal` slices the frames, then each is multiplied by a Hann window, and finally your hand-written `aurora_fft` (`aurora.audio.transforms.fft`, the Cooley-Tukey from L37-39) keeps the first `win_len//2+1` columns, giving a `(n_frames, win_len//2+1)` `complex128` matrix with numerical error under 1e-9. Then `np.abs()` squashes the complex matrix into a magnitude spectrum. The bigger `win_len`, the finer the frequency resolution, but the more time each frame covers — exactly the STFT's time–frequency trade-off. The next lesson (L45) builds a spectrogram from this complex matrix, drawing the time–frequency energy directly as a heatmap.


## ✏️ Closed-book derivation check cell — the STFT frame-count formula

**Rule: close every cell above and do the following derivation from memory alone.**

**Problem**: given:
- signal length $L$ (number of samples)
- window length $N_{fft}$
- hop $H$ (hop_length)
- center padding mode (pad $N_{fft}/2$ samples onto each end of the signal)

derive the formula for the number of frames $n_{frames}$ the STFT outputs.

Write out the formula and explain what each step means physically.

(Write your derivation here...)

### Going deeper: the STFT frame-count formula (the center=True case)

Before the closed-book derivation, let's work through it once in full so you can build the intuition.

**Given**:
- signal length: L (number of samples)
- FFT window length: N_fft (the frame length, e.g. 2048)
- hop: H (hop_length, e.g. 512)
- center=True: reflect-pad N_fft//2 samples onto each end

**Derivation**:

1. **Signal length after padding**:
   ```
   pad_left  = N_fft // 2
   pad_right = N_fft // 2
   padded_len = L + pad_left + pad_right = L + N_fft
   ```

2. **Where frame t sits** (in the padded signal):
   - start index: t × H
   - end index: t × H + N_fft - 1 (inclusive)
   - requirement for the last frame: `t × H + N_fft - 1 ≤ padded_len - 1`
   - simplify: `t × H + N_fft ≤ padded_len`
   - substitute `padded_len = L + N_fft`:
   ```
   t × H + N_fft ≤ L + N_fft
   t × H ≤ L
   t ≤ L // H (rounding down)
   ```

3. **Counting the frames**:
   - first frame index: t = 0
   - last frame index: t = L // H
   - **number of frames n_frames = (L // H) + 1**

**A small worked check** (L=100, H=10):
```
last frame index t_max = 100 // 10 = 10
n_frames = 10 + 1 = 11

Check:
  • frame 10 starts at: 10 × 10 = 100
  • frame 10 ends at:   100 + N_fft - 1
  • requirement: 100 + N_fft - 1 ≤ (100 + N_fft) - 1 ✓ (the edge case: frame 10 fits exactly)
  • frame 11 would start at 11 × 10 = 110 > L, out of range ✗
```

So the formula is: **n_frames = 1 + L // H**

**How this differs from center=False**:
- with center=False there's no padding and the start index is 0; the last frame needs `t×H + N_fft ≤ L`, giving `t ≤ (L - N_fft) // H`, so `n_frames = 1 + (L - N_fft) // H`
- with center=True the padded length becomes `L + N_fft`, so `n_frames = 1 + L // H` (quite a few more frames!)

Now for the closed-book derivation check:

In [ ]:
# Check: the formula agrees with the frame count from aurora.audio.stft
import numpy as np, sys
sys.path.insert(0, 'src')
from aurora.audio.stft import stft

SR, N_FFT, HOP = 16000, 1024, 256
signal = np.sin(2 * np.pi * 440 * np.arange(SR) / SR).astype(np.float32)

spec = stft(signal, n_fft=N_FFT, hop_length=HOP)
actual_frames = spec.shape[0]

# Formula: center=True → padded length L + N_FFT; n_frames = 1 + L // HOP
formula_frames = 1 + len(signal) // HOP
assert actual_frames == formula_frames, (
    f"the formula gives {formula_frames}, actual {actual_frames}: check your derivation"
)
print(f"✅ frame count checks out: n_frames = {actual_frames} (formula = {formula_frames})")

In [ ]:
# ✏️ Self-review for this lesson
l44_review = {
    "stft_three_steps":        None,  # remember the frame→window→fft three-step pipeline? True/False
    "my_stft_implemented":     None,  # my_stft implemented and passes the atol=1e-9 comparison? True/False
    "n_frames_formula":        None,  # derived n_frames=1+L//hop for center=True closed-book? True/False
    "win_vs_hop_tradeoff":     None,  # understand the trade-off win_len↑→finer frequency, hop↓→denser time? True/False
    "whiteboard_passed":       None,  # whiteboard challenge (frame-count derivation) done? True/False
    "single_frame_half_step":  None,  # single-frame STFT half-step matches aurora.stft[0]? True/False
}

unfilled = [k for k, v in l44_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l44_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L44 fully cleared! On to L45: drawing the STFT as a spectrogram')

---

→ **Next lesson** · [L45 · Making spectrograms](L45_spectrogram.ipynb)

> Next lesson covers **making spectrograms**: from the |STFT|² power spectrum to a dB heatmap — an X-ray of sound in time and frequency.